<a href="https://colab.research.google.com/github/MatinEbr/AdventureWorks/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [1]:
%cd /content
!rm -rf BasicSR Real-ESRGAN

# ۱. دانلود BasicSR (پیش‌نیاز اصلی) و رفع ارور locals در پایتون ۳.۱۳
!git clone https://github.com/XPixelGroup/BasicSR.git
%cd BasicSR
!sed -i 's/return locals()\[.__version__.\]/return "1.4.2"/g' setup.py
!sed -i 's/tb-nightly/tensorboard/g' requirements.txt
!pip install -e .

# ۲. دانلود Real-ESRGAN و رفع ارورهای مشابه
%cd /content
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!sed -i 's/return locals()\[.__version__.\]/return "0.3.0"/g' setup.py
!sed -i 's/torch>=1.7.0/torch/g' requirements.txt
!sed -i 's/torchvision>=0.8.1/torchvision/g' requirements.txt
!sed -i 's/numpy==1.21.1/numpy/g' requirements.txt

# نصب سایر پیش‌نیازها
!pip install facexlib gfpgan
!pip install -r requirements.txt
!pip install -e .

# ۳. رفع ارور توابع حذف‌شده‌ی PyTorch در فایل‌های BasicSR
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /content/BasicSR/basicsr/data/degradations.py

/content
Cloning into 'BasicSR'...
remote: Enumerating objects: 5924, done.
remote: Total 5924 (delta 0), reused 0 (delta 0), pack-reused 5924 (from 1)
Receiving objects: 100% (5924/5924), 4.14 MiB | 25.83 MiB/s, done.
Resolving deltas: 100% (3759/3759), done.
/content/BasicSR
Obtaining file:///content/BasicSR
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
  Building editable for basicsr (pyproject.toml) ... done
  Created wheel for basicsr: filename=basicsr-1.4.2-0.editable-py3-none-any.whl size=10625 sha256=badb3da4d99814059a9891244a191cd1dd2cb1608c9b5bcd8da34b11bd0e5a12
  Stored in dire

/content
Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 37.01 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/Real-ESRGAN
  Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4


Obtaining file:///content/Real-ESRGAN
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for realesrgan (pyproject.toml) ... done
  Created wheel for realesrgan: filename=realesrgan-0.3.0-0.editable-py3-none-any.whl size=9559 sha256=8cc741da794b0dabf04efe6646c315896a9942a5e8cdbc4c65228022478cab9a
  Stored in directory: /tmp/pip-ephem-wheel-cache-n_paluvp/wheels/1d/51/7e/125b783d94d3c0371e86bbf0f59cf7ed161995b7231bba6f90
Successfully built realesrgan
  Attempting uninstall: realesrgan
    Found existing installation: realesrgan 0.3.0
    Uninstalling realesrgan-0.3.0:
      Successfully uninstalled realesrgan-0.3.0



# 2. Mount drive (optional)

In [ ]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [ ]:
video_path="/content/video.mp4" #@param{type:"string"}
output_dir="/content/" #@param{type:"string"}
resolution = "2k (2560 x 1440)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

assert os.path.exists(video_path), "Video file does not exist"

video_capture = cv2.VideoCapture(video_path)
video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

final_width = None
final_height = None
aspect_ratio = float(video_width/video_height)

# Get output resolutions
match resolution:
  case "FHD (1920 x 1080)":
    final_width=1920
    final_height=1080
  case "2k (2560 x 1440)":
    final_width=2560
    final_height=1440
  case "4k (3840 x 2160)":
    final_width=3840
    final_height=2160
  case "2 x original":
    final_width=2*video_width
    final_height=2*video_height
  case "3 x original":
    final_width=3*video_width
    final_height=3*video_height
  case "4 x original":
    final_width=4*video_width
    final_height=4*video_height

if aspect_ratio == 1.0 and "original" not in resolution:
  final_height = final_width

if aspect_ratio < 1.0 and "original" not in resolution:
  temp = final_width
  final_width = final_height
  final_height = temp

scale_factor = max(final_width/video_width, final_height/video_height)
isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

# scale_factor needs to be even
while isEven == False:
  scale_factor += 0.01
  isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

print(f"Upscaling from {video_width}x{video_height} to {final_width}x{final_height}, scale_factor={scale_factor}")

!python inference_realesrgan_video.py -n {model} -i '{video_path}' -o '{output_dir}' --outscale {scale_factor}

video_name_with_ext = os.path.basename(video_path)
video_name = video_name_with_ext.replace(".mp4","")
upscaled_video_path = f"{output_dir}{video_name}_out.mp4"
final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"
final_video_path = os.path.join(output_dir, final_video_name)

# crop to fit
if "original" not in resolution:
  print("Cropping to fit...")
  command = f"ffmpeg -loglevel error -hwaccel cuda -y -i '{upscaled_video_path}' -c:v h264_nvenc -filter:v  'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
  subprocess.run(command,shell=True)
else:
  # final video path = upscaled video path
  command = f"cp '{upscaled_video_path}' '{final_video_path}'"
  subprocess.run(command,shell=True)

print(f"Upscaled video saved to: {final_video_path}")

# save to drive
if mount_drive:
  drive_folder = "MyDrive/Upscaled Videos (REAL-ESRGAN)"
  save_directory_drive = f"/content/gdrive/{drive_folder}"
  os.makedirs(save_directory_drive, exist_ok=True)

  command = f"cp '{final_video_path}' '{save_directory_drive}/{final_video_name}'"
  subprocess.run(command,shell=True)
  print(f"Saved to drive: /{drive_folder}/{final_video_name}" )

!rm "{upscaled_video_path}"

# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()

In [3]:
%cd /content/Real-ESRGAN
!python inference_realesrgan_video.py -i /content/vid.mp4 -n RealESRGAN_x4plus -s 2 -o /content/vid_out.mp4

/content/Real-ESRGAN
Traceback (most recent call last):
  File "/content/Real-ESRGAN/inference_realesrgan_video.py", line 398, in <module>
    main()
    ~~~~^^
  File "/content/Real-ESRGAN/inference_realesrgan_video.py", line 390, in main
    run(args)
    ~~~^^^^^^
  File "/content/Real-ESRGAN/inference_realesrgan_video.py", line 292, in run
    inference_video(args, video_save_path)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Real-ESRGAN/inference_realesrgan_video.py", line 249, in inference_video
    reader = Reader(args, total_workers, worker_idx)
  File "/content/Real-ESRGAN/inference_realesrgan_video.py", line 72, in __init__
    meta = get_video_meta_info(video_path)
  File "/content/Real-ESRGAN/inference_realesrgan_video.py", line 35, in get_video_meta_info
    ret['nb_frames'] = int(video_streams[0]['nb_frames'])
                           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
KeyError: 'nb_frames'
[vost#0:0/rawvideo @ 0x588115093300] Error submitting a packet to the 

**Reasoning**:
We will read the contents of `/content/Real-ESRGAN/inference_realesrgan_video.py` in python to locate `get_video_meta_info` and modify it to safely handle missing 'nb_frames' with a fallback using OpenCV or duration/fps.



In [4]:
import cv2
import os

file_path = '/content/Real-ESRGAN/inference_realesrgan_video.py'
with open(file_path, 'r') as f:
    content = f.read()

# Let's inspect get_video_meta_info function or perform inline replacement.
# The typical structure of get_video_meta_info uses ffprobe:
# ret['nb_frames'] = int(video_streams[0]['nb_frames'])

old_segment = "ret['nb_frames'] = int(video_streams[0]['nb_frames'])"
new_segment = """if 'nb_frames' in video_streams[0]:
        ret['nb_frames'] = int(video_streams[0]['nb_frames'])
    else:
        import cv2
        cap = cv2.VideoCapture(video_path)
        ret['nb_frames'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()"""

if old_segment in content:
    content = content.replace(old_segment, new_segment)
    with open(file_path, 'w') as f:
        f.write(content)
    print("Successfully patched get_video_meta_info in inference_realesrgan_video.py")
else:
    print("Target line not found directly. Let's find get_video_meta_info function manually.")
    # Let's search for nb_frames
    lines = content.splitlines()
    for i, line in enumerate(lines):
        if 'nb_frames' in line:
            print(f"Line {i}: {line}")

Successfully patched get_video_meta_info in inference_realesrgan_video.py


# Task
Modify "/content/Real-ESRGAN/inference_realesrgan_video.py" to safely handle videos missing the 'nb_frames' metadata key (using OpenCV or ffprobe total duration and frame rate fallback), and then run the upscale video command on "/content/vid.mp4" using the "RealESRGAN_x4plus" model with a scale factor of 2.

## بررسی و رفع باگ nb_frames

### Subtask:
Modify /content/Real-ESRGAN/inference_realesrgan_video.py to safely handle missing 'nb_frames' in ffprobe metadata using OpenCV or fallback calculation.


**Reasoning**:
Run the video upscaling command with python using the patched inference_realesrgan_video.py script to verify the fix works properly for /content/vid.mp4.



In [5]:
%cd /content/Real-ESRGAN
!python inference_realesrgan_video.py -i /content/vid.mp4 -n RealESRGAN_x4plus -s 2 -o /content/vid_out.mp4

/content/Real-ESRGAN
inference: 100% 482/482 [11:58<00:00,  1.49s/frame]


## اجرای تست ارتقای ویدیو

### Subtask:
Run the video upscaling command on the input video using the patched inference script to verify the fix.
